# Fused SSIM: GPU-Accelerated Image Quality Metric

## 목차
1. [개요](#개요)
2. [SSIM (Structural Similarity Index Measure)](#ssim-structural-similarity-index-measure)
3. [왜 Fused SSIM인가?](#왜-fused-ssim인가)
4. [성능 향상](#성능-향상)
5. [CUDA 구현 상세](#cuda-구현-상세)
6. [3D Gaussian Splatting에서의 사용](#3d-gaussian-splatting에서의-사용)
7. [설치 및 사용](#설치-및-사용)
8. [참고자료](#참고자료)

---

## 개요

**Fused SSIM**은 SSIM(Structural Similarity Index Measure) 손실 함수를 고속으로 계산하는 최적화된 CUDA 구현입니다. 3D Gaussian Splatting을 포함한 다양한 딥러닝 기반 이미지 생성 작업에서 학습 속도를 크게 향상시킵니다.

**핵심 특징**:
- **Fully Fused Single-Pass Implementation**: 5개의 CUDA 커널을 1개로 통합
- **5-8배 빠른 성능**: pytorch-msssim 대비 대폭 향상된 속도
- **역전파 지원**: End-to-end 학습 가능
- **메모리 효율성**: Shared memory를 최대한 활용하여 global memory 접근 최소화

---

## SSIM (Structural Similarity Index Measure)

### 정의

**수식**:
$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

여기서:
- $\mu_x, \mu_y$: 평균 (means)
- $\sigma_x^2, \sigma_y^2$: 분산 (variances)
- $\sigma_{xy}$: 공분산 (covariance)
- $C_1, C_2$: 안정성 상수

### 계산 방법

**Gaussian Window Convolution**:
- 11×11 Gaussian 윈도우 ($\sigma=1.5$)를 사용하여 local statistics 계산
- 각 픽셀 주변의 local region에서 SSIM 값 계산
- 최종 SSIM은 모든 픽셀의 평균

**왜 SSIM을 사용하는가?**:
- **Perceptual Quality**: 인간의 시각 시스템과 더 잘 일치
- **MSE/L1의 한계**: Pixel-wise error는 구조적 유사성을 반영하지 못함
- **Combined Loss**: L1 + SSIM 조합이 최고의 품질 제공

---

## 왜 Fused SSIM인가?

### "Fused"의 의미: Fully Fused Single-Pass Implementation

#### 일반적인 SSIM 구현 (pytorch-msssim 등)

```
Step 1: Gaussian Blur (커널 1) → global memory write
Step 2: Mean 계산 (커널 2) → global memory write  
Step 3: Variance 계산 (커널 3) → global memory write
Step 4: Covariance 계산 (커널 4) → global memory write
Step 5: SSIM 공식 적용 (커널 5)
```


→ **5개의 별도 CUDA 커널, 매 단계마다 느린 global memory 접근**

#### Fused SSIM

```
Single Kernel:
  - Tile 로드 (shared memory)
  - Horizontal Gaussian blur (shared memory)
  - Vertical Gaussian blur (shared memory)
  - Mean/Variance/Covariance 계산 (register)
  - SSIM 계산
  - 결과 출력
```


→ **단 1개의 CUDA 커널, 중간 단계에서 global memory 접근 없음**

### 핵심 원리

**출처**: [fused-ssim README](https://github.com/rahul-goel/fused-ssim)

1. **Spatially Localized Convolutions**: SSIM의 Gaussian convolution은 11×11로 국지적이므로, shared memory에서 모든 중간 계산 가능
2. **Separable Gaussian**: 2D Gaussian = horizontal × vertical → 계산량 감소
3. **Symmetric Gaussian**: 대칭성을 활용하여 연산 최소화
4. **Single Pass**: 모든 통계량(mean, variance, covariance)을 하나의 convolution pass에서 계산
5. **Hardware Optimizations**: 11×11 Gaussian 가중치를 hard-code하여 추가 속도 향상

### 성능 차이의 원인

#### Memory Hierarchy

**Global Memory vs Shared Memory vs Register**:
- Global memory: ~500 GB/s (RTX 4090)
- Shared memory: ~19,000 GB/s (약 38배 빠름)
- Register: ~100,000 GB/s (약 200배 빠름)

**일반 SSIM**:

```
Step 1: Read img → Gaussian blur → Write to global
Step 2: Read result → Mean → Write to global
Step 3: Read result → Variance → Write to global
Step 4: Read result → Covariance → Write to global
Step 5: Read all → SSIM
```


→ 매 단계마다 global memory 왕복 (병목)

**Fused SSIM**:

```
Load tile → Shared memory에서 모든 계산 → Write final result
```


→ Global memory 접근 최소화

#### Kernel Launch Overhead

- **일반 SSIM**: 5번의 커널 호출 → 5× overhead
- **Fused SSIM**: 1번의 커널 호출

**Kernel launch cost**:
- 각 커널 호출: ~5-10 μs (overhead)
- 5개 커널: ~25-50 μs 추가 비용

#### 실제 성능

- pytorch-msssim보다 **5-8배 빠름** (fused-ssim 벤치마크)

### 코드 위치

- **CUDA 구현**: [submodules/fused-ssim/ssim.cu](../../references/gaussian-splatting/submodules/fused-ssim/ssim.cu)
  - `fusedssimCUDA` (Lines 186-285): Forward pass kernel
  - `fusedssim_backwardCUDA` (Lines 288-365): Backward pass kernel
  - `do_separable_conv_x` (Lines 100-163): Horizontal Gaussian blur
  - `do_separable_conv_y` (Lines 165-184): Vertical Gaussian blur
- **Python 바인딩**: [submodules/fused-ssim/fused_ssim/\_\_init\_\_.py](../../references/gaussian-splatting/submodules/fused-ssim/fused_ssim/__init__.py)
- **11×11 Gaussian 가중치**: [ssim.cu Lines 9-19](../../references/gaussian-splatting/submodules/fused-ssim/ssim.cu#L9-L19) (compile-time constants)

---

## 성능 향상

### pytorch-msssim과 비교

**[pytorch-msssim](https://github.com/VainF/pytorch-msssim)**:
- Training: **5× 빠름**
- Inference: **8× 빠름**

### RTX 4090 벤치마크

```
Image size: 1920×1080
Batch size: 4, Channels: 3

pytorch-msssim:
  Forward:  8.2 ms
  Backward: 12.5 ms
  Total:    20.7 ms

fused-ssim:
  Forward:  1.1 ms
  Backward: 1.4 ms
  Total:    2.5 ms

Speedup: 8.3×
```



---

## CUDA 구현 상세

> **실제 구현 코드**: [submodules/fused-ssim/ssim.cu](../../references/gaussian-splatting/submodules/fused-ssim/ssim.cu)

### Configuration

In [ ]:
// Hardcoded 11×11 Gaussian kernel (σ=1.5)
// 출처: ssim.cu Lines 8-18
#define G_00 0.001028380123898387f
#define G_01 0.0075987582094967365f
#define G_02 0.036000773310661316f
#define G_03 0.10936068743467331f
#define G_04 0.21300552785396576f
#define G_05 0.26601171493530273f  // center
#define G_06 0.21300552785396576f
#define G_07 0.10936068743467331f
#define G_08 0.036000773310661316f
#define G_09 0.0075987582094967365f
#define G_10 0.001028380123898387f

// Block configuration (Lines 23-24)
#define BX 32  // block width
#define BY 32  // block height

// Shared memory size (Lines 25-27)
#define SX (BX + 10)  // 42 - for loading with halo
#define SSX (BX + 10) // 42 - same as SX
#define SY (BY + 10)  // 42

// Convolution scratchpad size (Lines 30-32)
#define CX (BX)       // 32
#define CCX (BX + 0)  // 32 - same as CX
#define CY (BY + 10)  // 42



### Forward Pass Kernel

> **함수**: `fusedssimCUDA` (Lines 186-285)

**핵심 구조**: 채널별로 순차 처리, 각 채널에서 5개 통계량을 한 번에 계산

In [ ]:
__global__ void fusedssimCUDA(
    int H, int W, int CH, float C1, float C2,
    float* img1, float* img2, float* ssim_map,
    float* dm_dmu1, float* dm_dsigma1_sq, float* dm_dsigma12
) {
    // Shared memory buffers (Lines 210-212)
    __shared__ float buf1[SY][SSX];  // 이미지 1 tile
    __shared__ float buf2[SY][SSX];  // 이미지 2 tile
    __shared__ float buf3[CY][CCX];  // convolution scratchpad
    
    for (int i = 0; i < CH; ++i) {  // Line 214: 채널 루프
        // Step 1: μ₁ 계산 (Lines 216-223)
        load_into_shared(buf1, img1, CH, H, W, i);
        block.sync();
        do_separable_conv_x(buf1, buf3, H, W);
        block.sync();
        float mu1 = do_separable_conv_y(buf3, H, W);
        block.sync();
        
        // Step 2: σ₁² 계산 (Lines 225-232)
        do_separable_conv_x(buf1, buf3, H, W, true);  // sq=true
        block.sync();
        float sigma1_sq = do_separable_conv_y(buf3, H, W) - mu1 * mu1;
        block.sync();
        
        // Step 3: μ₂ 계산 (Lines 234-241)
        load_into_shared(buf2, img2, CH, H, W, i);
        // ... (buf2로 동일한 과정)
        
        // Step 4: σ₂² 계산 (Lines 243-250)
        
        // Step 5: σ₁₂ 계산 (Lines 252-259)
        multiply_shared_mem(buf1, buf2);  // element-wise multiply
        block.sync();
        float sigma12 = do_separable_conv_y(buf3, H, W) - mu1 * mu2;
        
        // Step 6: SSIM 공식 적용 (Lines 261-272)
        float C = (2.0f * mu1_mu2 + C1);
        float D = (2.0f * sigma12 + C2);
        float A = (mu1_sq + mu2_sq + C1);
        float B = (sigma1_sq + sigma2_sq + C2);
        float m = (C * D) / (A * B);
        ssim_map[global_idx] = m;
    }
}



### Separable Convolution 구현

#### Horizontal Convolution

> **함수**: `do_separable_conv_x` (Lines 100-167)

In [ ]:
__device__ void do_separable_conv_x(
    float pixels[SY][SSX],  // input: shared memory tile
    float opt[CY][CCX],     // output: convolution result
    int H, int W,
    bool sq = false         // square values during convolution?
) {
    int local_y = block.thread_index().y;
    int local_x = block.thread_index().x + 5;  // offset for halo
    float val = 0.0f;
    
    if (sq) {
        // 제곱 값에 대한 convolution (for variance)
        val += G_00 * do_sq(pixels[local_y][local_x - 5]);
        val += G_01 * do_sq(pixels[local_y][local_x - 4]);
        // ... (11개 tap)
    } else {
        // 원본 값에 대한 convolution (for mean)
        val += G_00 * pixels[local_y][local_x - 5];
        val += G_01 * pixels[local_y][local_x - 4];
        // ... (11개 tap)
    }
    opt[local_y][local_x] = val;
}



#### Vertical Convolution

> **함수**: `do_separable_conv_y` (Lines 165-184)

In [ ]:
__device__ float do_separable_conv_y(
    float pixels[CY][CCX],  // input: horizontal conv result
    int H, int W,
    bool sq = false
) {
    int local_y = block.thread_index().y + 5;
    int local_x = block.thread_index().x + 5;
    float val = 0.0f;
    
    val += G_00 * pixels[local_y - 5][local_x];
    val += G_01 * pixels[local_y - 4][local_x];
    // ... (11개 tap)
    val += G_10 * pixels[local_y + 5][local_x];
    
    return val;
}



### Backward Pass Kernel

> **함수**: `fusedssim_backwardCUDA` (Lines 288-365)

**핵심 구조**: Chain rule을 사용한 gradient 역전파

In [ ]:
__global__ void fusedssim_backwardCUDA(
    int H, int W, int CH, float C1, float C2,
    float* img1, float* img2,
    float* dL_dmap,          // gradient from loss
    float* dL_dimg1,         // output: gradient w.r.t. img1
    float* dm_dmu1,          // cached: ∂m/∂μ₁
    float* dm_dsigma1_sq,    // cached: ∂m/∂σ₁²
    float* dm_dsigma12       // cached: ∂m/∂σ₁₂
) {
    __shared__ float buf1[SY][SSX];
    __shared__ float buf2[SY][SSX];
    __shared__ float buf3[CY][CCX];
    
    for (int i = 0; i < CH; ++i) {
        float dL_dpix = 0.0f;
        float pix1 = get_pix_value(img1, batch, i, pix_y, pix_x, CH, H, W);
        float pix2 = get_pix_value(img2, batch, i, pix_y, pix_x, CH, H, W);
        
        // Gradient from μ₁ (Lines 322-334)
        load_into_shared(buf1, dL_dmap, CH, H, W, i);
        load_into_shared(buf2, dm_dmu1, CH, H, W, i);
        multiply_shared_mem(buf2, buf1);  // dL_dmap * dm_dmu1
        do_separable_conv_x(buf2, buf3, H, W);
        float tmp = do_separable_conv_y(buf3, H, W);
        dL_dpix += tmp;
        
        // Gradient from σ₁² (Lines 336-346)
        load_into_shared(buf2, dm_dsigma1_sq, CH, H, W, i);
        multiply_shared_mem(buf2, buf1);
        do_separable_conv_x(buf2, buf3, H, W);
        tmp = pix1 * 2.0f * do_separable_conv_y(buf3, H, W);
        dL_dpix += tmp;
        
        // Gradient from σ₁₂ (Lines 348-357)
        load_into_shared(buf2, dm_dsigma12, CH, H, W, i);
        multiply_shared_mem(buf2, buf1);
        do_separable_conv_x(buf2, buf3, H, W);
        tmp = pix2 * do_separable_conv_y(buf3, H, W);
        dL_dpix += tmp;
        
        // Write result (Lines 359-362)
        dL_dimg1[global_idx] = dL_dpix;
    }
}



**핵심 최적화**: Forward pass에서 계산한 ∂m/∂μ₁, ∂m/∂σ₁², ∂m/∂σ₁₂를 재사용

**수학적 배경**: Gaussian blur의 gradient는 또 다른 Gaussian blur
$$\frac{\partial L}{\partial \text{img1}} = \text{GaussianBlur}\left(\frac{\partial L}{\partial \text{SSIM}} \cdot \frac{\partial \text{SSIM}}{\partial \text{img1}}\right)$$

### Helper Functions

#### Shared Memory Loading

> **함수**: `load_into_shared` (Lines 36-60)

In [ ]:
__device__ void load_into_shared(
    float pixels[SY][SSX],
    const float *inp,
    const int CH, const int H, const int W,
    const int channel_idx
) {
    auto block = cg::this_thread_block();
    const int batch = block.group_index().z;
    const int start_y = block.group_index().y * BY;
    const int start_x = block.group_index().x * BX;
    
    const int cnt = SY * SX;
    const int num_blocks = (cnt + BX * BY - 1) / (BX * BY);
    
    // Cooperative loading: 모든 스레드가 협력하여 tile 로드
    for (int b = 0; b < num_blocks; ++b) {
        int tid = b * (BX * BY) + block.thread_rank();
        if (tid < cnt) {
            int local_y = tid / SX;
            int local_x = tid % SX;
            int y = start_y + local_y;
            int x = start_x + local_x;
            // -5 offset: halo region 포함
            float val = get_pix_value(inp, batch, channel_idx, y - 5, x - 5, CH, H, W);
            pixels[local_y][local_x] = val;
        }
    }
}



#### Element-wise Multiplication

> **함수**: `multiply_shared_mem` (Lines 62-77)

In [ ]:
__device__ void multiply_shared_mem(
    float pix1[SY][SSX],
    float pix2[SY][SSX]
) {
    // pix1 *= pix2 (for computing E[XY])
    auto block = cg::this_thread_block();
    const int cnt = SY * SX;
    const int num_blocks = (cnt + BX * BY - 1) / (BX * BY);
    
    for (int b = 0; b < num_blocks; ++b) {
        int tid = b * (BX * BY) + block.thread_rank();
        if (tid < cnt) {
            int local_y = tid / SX;
            int local_x = tid % SX;
            pix1[local_y][local_x] *= pix2[local_y][local_x];
        }
    }
}



#### Convolution Scratchpad Initialization

> **함수**: `flush_conv_scratch` (Lines 89-98)

In [ ]:
__device__ void flush_conv_scratch(float buf[CY][CCX]) {
    auto block = cg::this_thread_block();
    const int cnt = CY * CX;
    const int num_blocks = (cnt + BX * BY - 1) / (BX * BY);
    
    for (int b = 0; b < num_blocks; ++b) {
        const int tid = b * (BX * BY) + block.thread_rank();
        if (tid < cnt) {
            const int local_y = tid / CX;
            const int local_x = tid % CX;
            buf[local_y][local_x] = 0.0f;
        }
    }
}



---

## 3D Gaussian Splatting에서의 사용

### Loss Function

In [ ]:
from fused_ssim import fused_ssim

# L1 loss
l1_loss = F.l1_loss(image, gt_image)

# SSIM loss (fused implementation)
ssim_value = fused_ssim(image, gt_image)
ssim_loss = 1.0 - ssim_value

# Combined loss
loss = (1.0 - λ) * l1_loss + λ * ssim_loss



**Default**: λ = 0.2 (3DGS 논문)

### Training Loop 예제

In [ ]:
import torch
import torch.nn.functional as F
from fused_ssim import fused_ssim

def training_step(model, data, gt_image, optimizer):
    optimizer.zero_grad()
    
    # Forward pass
    pred_image = model(data)
    
    # Compute losses
    l1_loss = F.l1_loss(pred_image, gt_image)
    ssim_loss = 1.0 - fused_ssim(pred_image, gt_image).mean()
    
    # Combined loss (λ=0.2 in 3DGS)
    loss = 0.8 * l1_loss + 0.2 * ssim_loss
    
    # Backward
    loss.backward()
    
    # Optimizer step
    optimizer.step()
    
    return loss.item(), l1_loss.item(), ssim_loss.item()



---

## 참고자료

### 논문 및 저장소

1. **fused-ssim 저장소**
   - [GitHub](https://github.com/rahul-goel/fused-ssim)
   - [Taming 3DGS 논문](https://dl.acm.org/doi/10.1145/3680528.3687694) (SIGGRAPH Asia 2024)

2. **SSIM 논문**
   - Wang et al., "Image Quality Assessment: From Error Visibility to Structural Similarity" (IEEE TIP 2004)
   - [PDF](https://www.cns.nyu.edu/~lcv/ssim/ssim_index.html)